# 16a — UR5e kinematic calibration and identifiability

**Outcomes:** fit four identifiable geometric/encoder corrections, distinguish
training loss from held-out accuracy, inspect scaled parameter sensitivity,
and carry calibrated FK into estimation and control. Prerequisite:
[16](16_ur5e_kinematics.ipynb). All definitions are repeated locally so a fresh
kernel can run this notebook without executing a sibling notebook.

### Prepare the notebook and optional interactive frontend

Imports below provide array operations, CasADi graphs, and presentation only.
Every robot model and optimization formulation is defined in this notebook.
Plotly gives hover/zoom plots when installed; Matplotlib is the fallback.
Widgets use explicit **Run** and **Reset** buttons, so moving a slider cannot
launch a solve. `CASADI_TUTORIAL_WIDGETS=0` disables widgets for batch execution.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

repository_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_robot, plot_series
from casadi_tutorial.diagnostics import bound_report, max_bound_violation, solver_summary

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)

### Write the robot model: frames, parameters, and coordinates

The table below is the [UR5e standard DH geometry published by Universal Robots](https://www.universal-robots.com/articles/ur/application-installation/dh-parameters-for-calculations-of-kinematics-and-dynamics/).
Rows follow joints 1–6 from base to tool; all lengths are metres and all angles are radians.

| Joint | $a_i$ | $d_i$ | $\alpha_i$ |
|---|---:|---:|---:|
| 1 | 0 | 0.1625 | $\pi/2$ |
| 2 | −0.425 | 0 | 0 |
| 3 | −0.3922 | 0 | 0 |
| 4 | 0 | 0.1333 | $\pi/2$ |
| 5 | 0 | 0.0997 | $-\pi/2$ |
| 6 | 0 | 0.0996 | 0 |

For **standard** DH, the relative transform is

$$
{}^{i-1}T_i=\begin{bmatrix}
c_i&-s_i c_{\alpha_i}&s_i s_{\alpha_i}&a_i c_i\\
s_i&c_i c_{\alpha_i}&-c_i s_{\alpha_i}&a_i s_i\\
0&s_{\alpha_i}&c_{\alpha_i}&d_i\\0&0&0&1
\end{bmatrix},\qquad
T^0_6=\prod_{i=1}^6{}^{i-1}T_i=\begin{bmatrix}R&p\\0&1\end{bmatrix}.
$$

Calibration will adjust only $\theta=[\Delta a_2,\Delta a_3,\delta_2,\delta_3]$.
Here $q$ always means the six **encoder coordinates**, and
$q_{\rm physical}=q+[0,\delta_2,\delta_3,0,0,0]^T$.
The offset is added exactly once, inside this model. Fixed base and tool frames are assumed known.
`pose_map(q, theta)` returns $4\times4$; `position_map` returns $3\times1$;
`origins_map` returns $3\times7$. Joint, speed, and acceleration limits below are conservative teaching choices.
These functions are defined locally so the whole graph can be inspected and edited here.

In [ ]:
# Standard DH, metres and radians. The table is included here rather than imported.
dh_a = np.array([0.0, -0.425, -0.3922, 0.0, 0.0, 0.0])
dh_d = np.array([0.1625, 0.0, 0.0, 0.1333, 0.0997, 0.0996])
dh_alpha = np.array([np.pi / 2, 0.0, 0.0, np.pi / 2, -np.pi / 2, 0.0])
q_min_physical = np.full(6, -2 * np.pi)
q_max_physical = np.full(6, 2 * np.pi)
v_max = np.full(6, np.pi)
u_max = np.full(6, 2 * np.pi)


def encoder_bias(theta):
    """theta = [delta_a2, delta_a3, delta_encoder2, delta_encoder3]."""
    return ca.vertcat(0, theta[2], theta[3], 0, 0, 0)


def dh_transform(angle, distance, length, twist):
    """One STANDARD-DH transform RotZ(angle) TransZ(d) TransX(a) RotX(alpha)."""
    c, s = ca.cos(angle), ca.sin(angle)
    ct, st = np.cos(twist), np.sin(twist)
    return ca.vertcat(
        ca.horzcat(c, -s * ct, s * st, length * c),
        ca.horzcat(s, c * ct, -c * st, length * s),
        ca.horzcat(0, st, ct, distance),
        ca.horzcat(0, 0, 0, 1),
    )


def link_transforms(q_encoder, theta=None):
    """Seven base-to-frame transforms, including the identity base frame."""
    theta = ca.DM.zeros(4) if theta is None else theta
    physical_q = q_encoder + encoder_bias(theta)
    lengths = ca.vertcat(dh_a[0], dh_a[1] + theta[0], dh_a[2] + theta[1], *dh_a[3:])
    transforms = [ca.DM.eye(4)]
    for joint in range(6):
        transforms.append(
            transforms[-1] @ dh_transform(
                physical_q[joint], dh_d[joint], lengths[joint], dh_alpha[joint]
            )
        )
    return transforms


def forward_kinematics(q_encoder, theta=None):
    return link_transforms(q_encoder, theta)[-1]


def frame_origins(q_encoder, theta=None):
    return ca.horzcat(*[transform[:3, 3] for transform in link_transforms(q_encoder, theta)])


q_symbol = ca.SX.sym("q_encoder", 6)
theta_symbol = ca.SX.sym("theta", 4)
pose_map = ca.Function("pose", [q_symbol, theta_symbol], [forward_kinematics(q_symbol, theta_symbol)])
position_map = ca.Function("position", [q_symbol, theta_symbol], [pose_map(q_symbol, theta_symbol)[:3, 3]])
origins_map = ca.Function("origins", [q_symbol, theta_symbol], [frame_origins(q_symbol, theta_symbol)])
nominal_theta = np.zeros(4)

### Create independent pose measurements and reserve validation data

A synthetic tracker observes the complete tool pose at 36 diverse, known encoder
configurations. The simulated plant has 4 mm and −3 mm link corrections and small
joint-2/3 zero offsets. **Only data generation sees `theta_truth`.** The estimator
starts at nominal parameters. Tracker noise is 0.5 mm per Cartesian component
and 0.002 rad per rotation-vector component.

Twenty-four randomly selected poses train the estimator; the remaining twelve
are held out. The split and random seed stay fixed when changing noise, allowing
fair comparisons. Tool/base extrinsics, other DH constants, and encoder noise
during this static calibration experiment are assumed known. Estimating every
DH entry and frame offset together would introduce ambiguity.

In [ ]:
from scipy.spatial.transform import Rotation

parameter_scale = np.array([0.01, 0.01, 0.05, 0.05])
theta_truth = np.array([0.004, -0.003, 0.015, -0.012])
sigma_position = 0.0005
sigma_rotation = 0.002


def calibration_dataset(noise_multiplier=1.0):
    rng = np.random.default_rng(2026)
    configurations = rng.uniform(
        [-1.5, -1.5, 0.2, -1.0, 0.4, -1.5],
        [1.5, -0.2, 1.7, 1.0, 1.6, 1.5], (36, 6),
    )
    measured = np.array([np.asarray(pose_map(q, theta_truth)) for q in configurations])
    measured[:, :3, 3] += rng.normal(0, sigma_position * noise_multiplier, (36, 3))
    for transform in measured:
        rotation_noise = Rotation.from_rotvec(rng.normal(0, sigma_rotation * noise_multiplier, 3)).as_matrix()
        transform[:3, :3] = transform[:3, :3] @ rotation_noise
    indices = rng.permutation(36)
    return {
        "q": configurations, "poses": measured,
        "training": indices[:24], "validation": indices[24:],
        "noise_multiplier": noise_multiplier,
    }


calibration_data = calibration_dataset()
print("Training / validation poses:", len(calibration_data["training"]), "/", len(calibration_data["validation"]))

### Formulate and solve a scaled nonlinear least-squares calibration

With $\theta=D z$, $D=\operatorname{diag}(0.01\,\mathrm m,0.01\,\mathrm m,
0.05\,\mathrm{rad},0.05\,\mathrm{rad})$, solve

$$\min_{-1\le z\le1}\sum_{j\in\mathcal T}\left\|
\begin{bmatrix}
(p(q_j;Dz)-p_j)/\sigma_p\\
\operatorname{vec}(R_j^TR(q_j;Dz)-I)/(\sqrt2\sigma_R)
\end{bmatrix}\right\|_2^2.$$

The rotation term is a smooth **chordal rotation residual**:
$\|R_j^TR-I\|_F^2/2=4\sin^2(\phi/2)\approx\phi^2$ for small relative angle
$\phi$. Its nine entries encode three rotation degrees of freedom; this is a
weighted least-squares teaching objective, not an exact independent-component likelihood.
Small, bounded corrections keep the problem close to the measured frame convention.
The $12|\mathcal T|\times4$ residual Jacobian is differentiated with respect to
scaled parameters so length and angle columns can be compared sensibly.

In [ ]:
def calibration_residual(q_observed, pose_observed, theta, noise_multiplier=1.0):
    prediction = pose_map(q_observed, theta)
    position_residual = (prediction[:3, 3] - pose_observed[:3, 3]) / (sigma_position * noise_multiplier)
    relative_rotation = pose_observed[:3, :3].T @ prediction[:3, :3]
    rotation_residual = ca.vec(relative_rotation - ca.DM.eye(3)) / (np.sqrt(2) * sigma_rotation * noise_multiplier)
    return ca.vertcat(position_residual, rotation_residual)


def fit_calibration(data, training_count=24):
    problem = ca.Opti()
    normalized_theta = problem.variable(4)
    theta = ca.DM(parameter_scale) * normalized_theta
    residual = ca.vertcat(*[
        calibration_residual(data["q"][index], ca.DM(data["poses"][index]), theta, data["noise_multiplier"])
        for index in data["training"][:training_count]
    ])
    problem.minimize(ca.sumsqr(residual))
    problem.subject_to(problem.bounded(-1, normalized_theta, 1))
    problem.set_initial(normalized_theta, 0)
    problem.solver("ipopt", {"print_time": False}, {"print_level": 0, "sb": "yes", "max_iter": 100, "tol": 1e-9})
    solution = problem.solve()
    estimated_theta = np.asarray(solution.value(theta)).ravel()
    sensitivity = ca.Function("calibration_sensitivity", [normalized_theta], [ca.jacobian(residual, normalized_theta)])
    singular_values = np.linalg.svd(np.asarray(sensitivity(estimated_theta / parameter_scale)), compute_uv=False)
    return {"theta": estimated_theta, "singular_values": singular_values, "stats": solution.stats(), "training_count": training_count}


calibration_result = fit_calibration(calibration_data)
theta_estimate = calibration_result["theta"]
print("[delta a2 [m], delta a3 [m], delta2 [rad], delta3 [rad]]")
print("estimated:", theta_estimate)
print("truth (evaluation only):", theta_truth)

### Validate prediction and identifiability on data the solver did not see

Report position and rotation-angle RMSE independently on training and held-out
poses. Good training loss alone can conceal a poor model. The scaled sensitivity
must have four non-negligible singular values; otherwise some parameter combination
cannot be resolved by these poses. The default held-out position error should
fall from roughly 10 mm to roughly 1 mm, near the synthetic noise level.

Parameter-recovery assertions below use simulation truth **only after fitting**.
Real calibration must instead rely on held-out measurements, uncertainty analysis,
and independent geometric checks. Bounds alone do not make a parameter identifiable.

In [ ]:
def pose_errors(data, theta, indices):
    position_errors, rotation_errors = [], []
    for index in indices:
        prediction = np.asarray(pose_map(data["q"][index], theta))
        measurement = data["poses"][index]
        position_errors.append(np.linalg.norm(prediction[:3, 3] - measurement[:3, 3]))
        rotation_errors.append(Rotation.from_matrix(measurement[:3, :3].T @ prediction[:3, :3]).magnitude())
    return np.asarray(position_errors), np.asarray(rotation_errors)


def summarize_calibration(data, result):
    metrics = {}
    for split in ("training", "validation"):
        indices = data[split][:result["training_count"]] if split == "training" else data[split]
        for label, theta in (("nominal", nominal_theta), ("calibrated", result["theta"])):
            position_error, rotation_error = pose_errors(data, theta, indices)
            metrics[split, label] = (np.sqrt(np.mean(position_error**2)), np.sqrt(np.mean(rotation_error**2)))
            print(f"{split:10s} {label:10s}: position RMSE {metrics[split, label][0] * 1000:.3f} mm; orientation RMSE {metrics[split, label][1]:.5f} rad")
    singular_values = result["singular_values"]
    print("Scaled sensitivity singular values:", singular_values)
    print("Scaled sensitivity condition number:", singular_values[0] / singular_values[-1])
    return metrics


calibration_metrics = summarize_calibration(calibration_data, calibration_result)
assert calibration_result["stats"]["success"]
assert calibration_result["singular_values"][-1] > 1e-6 * calibration_result["singular_values"][0]
assert calibration_metrics["validation", "calibrated"][0] < 0.002
assert calibration_metrics["validation", "calibrated"][0] < calibration_metrics["validation", "nominal"][0]
np.testing.assert_allclose(theta_estimate[:2], theta_truth[:2], atol=0.001)
np.testing.assert_allclose(theta_estimate[2:], theta_truth[2:], atol=0.002)

### Visualize held-out errors and an unobservable calibration direction

Each point below is a held-out pose. Hover to compare nominal and calibrated
position errors. Separately, differentiate tool position with respect to joint
6's zero offset. This column is identically zero: rotating the last joint
changes orientation but not this DH tool origin. Position-only data cannot
recover that offset. Our four-parameter choice avoids this specific ambiguity;
fitting arbitrary base/tool frames would introduce additional ones.

In [ ]:
nominal_error, _ = pose_errors(calibration_data, nominal_theta, calibration_data["validation"])
fitted_error, _ = pose_errors(calibration_data, theta_estimate, calibration_data["validation"])
display_plot(plot_series(np.arange(1, 13), {"nominal": nominal_error * 1000, "calibrated": fitted_error * 1000}, title="Held-out tool-position errors", ylabel="position error [mm]"))
sixth_offset = ca.SX.sym("sixth_offset")
shifted_q = q_symbol + ca.vertcat(0, 0, 0, 0, 0, sixth_offset)
sixth_sensitivity = ca.Function("sixth_offset_position_sensitivity", [q_symbol, sixth_offset], [ca.jacobian(position_map(shifted_q, nominal_theta), sixth_offset)])
np.testing.assert_allclose(sixth_sensitivity(calibration_data["q"][0], 0.0), np.zeros((3, 1)), atol=1e-12)
print("Tool-position sensitivity to joint-6 zero offset:", np.asarray(sixth_sensitivity(calibration_data["q"][0], 0.0)).ravel())

### Experiment with noise and training-pose count

Change the sensor noise multiplier or reduce the training set, then press
**Run**. The twelve validation poses stay held out. Inspect the smallest
singular value and validation error together: more optimizer iterations cannot
repair uninformative measurements. The callback intentionally fits only after
the button is pressed. Default results above remain available for comparison.

In [ ]:
def calibration_experiment(noise_multiplier=1.0, training_count=24):
    data = calibration_dataset(float(noise_multiplier))
    result = fit_calibration(data, int(training_count))
    summarize_calibration(data, result)
    nominal_errors, _ = pose_errors(data, nominal_theta, data["validation"])
    fitted_errors, _ = pose_errors(data, result["theta"], data["validation"])
    display_plot(plot_series(np.arange(1, 13), {"nominal": nominal_errors * 1000, "calibrated": fitted_errors * 1000}, title="Experiment: held-out calibration errors", ylabel="error [mm]"))


controls = {
    "noise_multiplier": widgets.FloatSlider(value=1, min=0.5, max=4, step=0.5, description="Noise ×", continuous_update=False) if widgets else 1.0,
    "training_count": widgets.IntSlider(value=24, min=8, max=24, step=4, description="Train poses", continuous_update=False) if widgets else 24,
}
experiment(calibration_experiment, controls, title="Fit a new calibration experiment")

### What carries forward

Exporting four fitted numbers is different from hiding a formulation in a
package: subsequent notebooks repeat both the model and calibration problem.
Calibrated FK will map encoder states to tracker observations, Cartesian
targets, and obstacle distances. The state estimator must still estimate joint
velocity from time history. Continue to [16b](16b_ur5e_state_estimation.ipynb).